In [2]:
import pandas as pd

In [3]:
from nsepy import get_history as gh
import datetime as dt

In [4]:
stk_data = pd.read_csv("Tatacoffee13_21.csv")

In [5]:
stk_data=stk_data[["Open","High","Low","Close"]]
#stk_data.to_csv("Tatacoffee13_21.csv")

In [6]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (2225, 4)


In [7]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [8]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

1780
X_train length: (1780, 4)
X_test length: (445, 4)
y_train length: (1780, 4)
y_test length: (445, 4)


In [9]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [11]:
listt=["Close","High","Open","Low"]
#listt=["AQI_calculated","PM10","PM2.5","NOx","NO2","NO","NH3","SO2","CO",'year']


In [14]:
def combination_varmax(dataset, listt):
    print(listt)
    datasetTwo = dataset[listt]
    test_obs = 28
    train = datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
    
    from statsmodels.tsa.statespace.varmax import VARMAX
    import pandas as pd
    import numpy as np
    from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

    # Fit the VARMAX model
    model = VARMAX(train, order=(1,1))
    result = model.fit(disp=False)

    # Forecast
    pred = result.forecast(steps=test_obs)
    preds = pd.DataFrame(pred, columns=listt)
    preds.to_csv("varmax_forecasted_{}.csv".format(test_obs))

    # Evaluate
    rmse = np.sqrt(mean_squared_error(test, pred))
    mape = mean_absolute_percentage_error(test, pred)

    performance["Model"].append(listt)
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append("(1,1)")
    performance["Test"].append(test_obs)

    perf = pd.DataFrame(performance)
    return perf, result, pred

In [15]:
perf,result,pred=combination_varmax(data1,listt)

['Close', 'High', 'Open', 'Low']


C:\Users\babuk\anaconda3\envs\aiml\Lib\site-packages\statsmodels\tsa\statespace\varmax.py:160: EstimationWarning: Estimation of VARMA(p,q) models is not generically robust, due especially to identification issues.
  warn('Estimation of VARMA(p,q) models is not generically robust,'
C:\Users\babuk\anaconda3\envs\aiml\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [16]:
data1

,Open,High,Low,Close
0,0.849900,0.845408,0.856100,0.854203
1,0.856394,0.967399,0.861040,0.974481
2,0.988480,0.996439,0.984959,0.986240
3,0.985483,0.968105,0.960760,0.956749
4,0.955669,0.975319,0.955033,0.967132
...,...,...,...,...
2220,0.095842,0.096329,0.096510,0.097323
2221,0.097777,0.095745,0.096951,0.096041
2222,0.096466,0.093934,0.095252,0.094821
2223,0.094031,0.105047,0.093143,0.105673


In [17]:
perf

,Model,RMSE,MaPe,Lag,Test
0,"[Close, High, Open, Low]",0.016628,0.159941,"(1,1)",28


VARMAX stands for:

Vector AutoRegressive Moving Average model with eXogenous regressors

It’s a time series forecasting model used for multivariate data — that is, when you have two or more related time series that influence each other.

It is an extension of simpler models like AR, MA, and VAR.

🔍 1. Let’s recall the simpler models first
Model	Full name	Works with	Captures
AR	AutoRegressive	1 variable	Past values of the same variable
MA	Moving Average	1 variable	Past forecast errors
ARMA	AutoRegressive Moving Average	1 variable	Both AR and MA effects
ARIMA	AutoRegressive Integrated Moving Average	1 variable	ARMA + differencing
VAR	Vector AutoRegression	Multiple variables	Past values of all variables influence each other

So, VAR extends AR to multiple variables.

🧩 2. What VARMAX adds

VARMAX adds two features:

MA part (Moving Average) → handles noise or short-term shocks.

X part (Exogenous variables) → allows including external predictors that affect your series (like marketing spend, temperature, etc.).

⚙️ 3. VARMAX model structure

For example, if you have two time series —

Sales_t

Revenue_t

and maybe one external variable like Marketing_t,
then a VARMAX model could look like this:


Meaning of order=(1,1) in VARMAX

In VARMAX(order=(p, q)):

p = number of autoregressive (AR) lags

q = number of moving average (MA) lags

So order=(1,1) means:

The model looks back 1 time step for AR and 1 time step for MA.

Summary Table
Parameter	Stands for	Typical Range	Meaning
p	AR order	0–3	How many past time steps of each variable to include
q	MA order	0–3	How many past forecast errors to include

Common starting choices:

(1,0) → Pure autoregressive (like VAR)

(1,1) → AR + MA (balanced, standard start)

(2,1) → If there are longer time dependencies